# Ana Gomes · Databricks portfolio import

Run all cells in Databricks. This notebook includes the original synthetic fixture. The first code cell creates the project schema and managed input volume in the selected catalog. It rewrites only the project fixture files. Output tables are written by the original project pipeline.

Project source: https://github.com/anaoutro/circular-intelligence-databricks

Prepared for import on 5 October 2026. Execution in Databricks is pending; local syntax checks do not validate Spark/Delta runtime behavior.


In [ ]:
import re, json
default_catalog = spark.sql("SELECT current_catalog()").first()[0]
dbutils.widgets.text("catalog", default_catalog)
dbutils.widgets.text("schema", 'circular_portfolio')
catalog = dbutils.widgets.get("catalog")
schema_name = dbutils.widgets.get("schema")
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog), "Catalog must use a simple identifier"
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", schema_name), "Schema must use a simple identifier"
prefix = f"{catalog}.{schema_name}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {prefix}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {prefix}.input")
volume_root = f"/Volumes/{catalog}/{schema_name}/input"
fixture_csv = 'equipment_id,model,supplier,updated_at,condition_score,failure_probability,resale_price,refurbished_price,repair_cost,salvage_value,acquisition_credit,logistics_cost\nEQ-0001,Atlas 14,Northstar IT,2026-09-01T12:00:00Z,45,0.38,930,1460,690,100,150,60\nEQ-0002,Orion 15,Northstar IT,2026-09-02T12:00:00Z,48,0.27,980,1320,700,150,370,70\nEQ-0003,Lumen 13,Northstar IT,2026-09-03T12:00:00Z,35,0.18,740,1030,100,120,300,90\nEQ-0004,Atlas 14,Harbor Devices,2026-09-04T12:00:00Z,13,0.29,1030,1410,790,100,350,70\nEQ-0005,Orion 15,Harbor Devices,2026-09-05T12:00:00Z,65,0.38,850,1310,410,150,420,40\nEQ-0006,Lumen 13,Harbor Devices,2026-09-06T12:00:00Z,21,0.10,740,1110,450,120,180,30\nEQ-0007,Atlas 14,Meridian Systems,2026-09-07T12:00:00Z,37,0.50,980,1380,140,100,580,40\nEQ-0008,Orion 15,Meridian Systems,2026-09-08T12:00:00Z,74,0.07,850,1490,630,150,90,90\nEQ-0009,Lumen 13,Meridian Systems,2026-09-09T12:00:00Z,96,0.10,620,1200,320,120,390,40\nEQ-0010,Atlas 14,Northstar IT,2026-09-10T12:00:00Z,66,0.17,1040,1450,790,100,260,90\nEQ-0011,Orion 15,Northstar IT,2026-09-11T12:00:00Z,48,0.08,860,1310,600,150,550,60\nEQ-0012,Lumen 13,Northstar IT,2026-09-12T12:00:00Z,60,0.28,610,1020,540,120,140,50\nEQ-0013,Atlas 14,Harbor Devices,2026-09-13T12:00:00Z,72,0.09,920,1340,380,100,560,70\nEQ-0014,Orion 15,Harbor Devices,2026-09-14T12:00:00Z,48,0.47,920,1430,410,150,300,30\nEQ-0015,Lumen 13,Harbor Devices,2026-09-15T12:00:00Z,72,0.42,610,1030,210,120,610,50\nEQ-0016,Atlas 14,Meridian Systems,2026-09-16T12:00:00Z,78,0.43,1040,1470,190,100,170,50\nEQ-0017,Orion 15,Meridian Systems,2026-09-17T12:00:00Z,15,0.48,870,1330,140,150,220,40\nEQ-0018,Lumen 13,Meridian Systems,2026-09-18T12:00:00Z,98,0.45,640,1130,110,120,490,70\nEQ-0019,Atlas 14,Northstar IT,2026-09-19T12:00:00Z,89,0.03,1020,1480,270,100,600,40\nEQ-0020,Orion 15,Northstar IT,2026-09-20T12:00:00Z,27,0.10,960,1370,570,150,550,60\nEQ-0021,Lumen 13,Northstar IT,2026-09-21T12:00:00Z,19,0.18,630,1190,600,120,400,60\nEQ-0022,Atlas 14,Harbor Devices,2026-09-22T12:00:00Z,44,0.24,960,1390,680,100,120,60\nEQ-0023,Orion 15,Harbor Devices,2026-09-23T12:00:00Z,62,0.59,950,1420,290,150,350,50\nEQ-0024,Lumen 13,Harbor Devices,2026-09-24T12:00:00Z,56,0.19,760,1200,160,120,270,80\nEQ-0025,Atlas 14,Meridian Systems,2026-09-25T12:00:00Z,18,0.33,920,1490,730,100,380,90\nEQ-0026,Orion 15,Meridian Systems,2026-09-26T12:00:00Z,90,0.42,910,1460,180,150,240,90\nEQ-0027,Lumen 13,Meridian Systems,2026-09-27T12:00:00Z,77,0.28,720,1140,670,120,530,30\nEQ-0028,Atlas 14,Northstar IT,2026-09-28T12:00:00Z,15,0.05,960,1480,230,100,270,40\nEQ-0029,Orion 15,Northstar IT,2026-09-01T12:00:00Z,45,0.08,950,1460,490,150,40,70\nEQ-0030,Lumen 13,Northstar IT,2026-09-02T12:00:00Z,66,0.02,710,1150,540,120,580,30\nEQ-0031,Atlas 14,Harbor Devices,2026-09-03T12:00:00Z,56,0.46,1060,1330,320,100,440,90\nEQ-0032,Orion 15,Harbor Devices,2026-09-04T12:00:00Z,93,0.27,940,1480,820,150,140,80\nEQ-0033,Lumen 13,Harbor Devices,2026-09-05T12:00:00Z,29,0.50,790,1000,80,120,270,70\nEQ-0034,Atlas 14,Meridian Systems,2026-09-06T12:00:00Z,24,0.08,1070,1430,360,100,470,60\nEQ-0035,Orion 15,Meridian Systems,2026-09-07T12:00:00Z,21,0.33,950,1500,550,150,580,90\nEQ-0036,Lumen 13,Meridian Systems,2026-09-08T12:00:00Z,37,0.15,800,1040,510,120,410,90\nEQ-0037,Atlas 14,Northstar IT,2026-09-09T12:00:00Z,42,0.51,960,1360,430,100,480,30\nEQ-0038,Orion 15,Northstar IT,2026-09-10T12:00:00Z,74,0.47,920,1370,580,150,580,90\nEQ-0039,Lumen 13,Northstar IT,2026-09-11T12:00:00Z,37,0.19,640,1150,630,120,160,80\nEQ-0040,Atlas 14,Harbor Devices,2026-09-12T12:00:00Z,79,0.13,900,1330,390,100,170,30\nEQ-0041,Orion 15,Harbor Devices,2026-09-13T12:00:00Z,95,0.06,900,1370,300,150,220,90\nEQ-0042,Lumen 13,Harbor Devices,2026-09-14T12:00:00Z,65,0.49,650,1060,720,120,250,30\nEQ-0043,Atlas 14,Meridian Systems,2026-09-15T12:00:00Z,64,0.22,1070,1340,590,100,200,60\nEQ-0044,Orion 15,Meridian Systems,2026-09-16T12:00:00Z,31,0.02,870,1390,820,150,160,60\nEQ-0045,Lumen 13,Meridian Systems,2026-09-17T12:00:00Z,67,0.12,790,1130,670,120,60,30\nEQ-0046,Atlas 14,Northstar IT,2026-09-18T12:00:00Z,78,0.55,970,1470,110,100,100,30\nEQ-0047,Orion 15,Northstar IT,2026-09-19T12:00:00Z,52,0.03,850,1380,470,150,520,50\nEQ-0048,Lumen 13,Northstar IT,2026-09-20T12:00:00Z,82,0.06,700,1090,550,120,330,50\nEQ-0049,Atlas 14,Harbor Devices,2026-09-21T12:00:00Z,92,0.40,1000,1490,450,100,240,80\nEQ-0050,Orion 15,Harbor Devices,2026-09-22T12:00:00Z,10,0.09,980,1400,170,150,580,50\nEQ-0051,Lumen 13,Harbor Devices,2026-09-23T12:00:00Z,78,0.26,640,1010,310,120,650,60\nEQ-0052,Atlas 14,Meridian Systems,2026-09-24T12:00:00Z,12,0.42,1090,1480,400,100,360,70\nEQ-0053,Orion 15,Meridian Systems,2026-09-25T12:00:00Z,50,0.18,880,1310,270,150,380,80\nEQ-0054,Lumen 13,Meridian Systems,2026-09-26T12:00:00Z,57,0.28,670,1200,620,120,530,90\nEQ-0055,Atlas 14,Northstar IT,2026-09-27T12:00:00Z,28,0.19,900,1310,490,100,420,80\nEQ-0056,Orion 15,Northstar IT,2026-09-28T12:00:00Z,88,0.18,960,1480,580,150,370,90\nEQ-0057,Lumen 13,Northstar IT,2026-09-01T12:00:00Z,87,0.16,760,1090,570,120,300,60\nEQ-0058,Atlas 14,Harbor Devices,2026-09-02T12:00:00Z,27,0.44,930,1470,210,100,440,90\nEQ-0059,Orion 15,Harbor Devices,2026-09-03T12:00:00Z,80,0.42,890,1360,650,150,110,40\nEQ-0060,Lumen 13,Harbor Devices,2026-09-04T12:00:00Z,58,0.03,670,1070,470,120,170,50\nEQ-0061,Atlas 14,Meridian Systems,2026-09-05T12:00:00Z,76,0.10,1000,1470,460,100,350,70\nEQ-0062,Orion 15,Meridian Systems,2026-09-06T12:00:00Z,94,0.26,900,1420,370,150,610,90\nEQ-0063,Lumen 13,Meridian Systems,2026-09-07T12:00:00Z,50,0.40,610,1160,610,120,470,50\nEQ-0064,Atlas 14,Northstar IT,2026-09-08T12:00:00Z,61,0.47,1010,1390,340,100,490,60\nEQ-0065,Orion 15,Northstar IT,2026-09-09T12:00:00Z,83,0.28,990,1410,490,150,200,30\nEQ-0066,Lumen 13,Northstar IT,2026-09-10T12:00:00Z,36,0.13,800,1110,290,120,370,40\nEQ-0067,Atlas 14,Harbor Devices,2026-09-11T12:00:00Z,39,0.40,1000,1480,310,100,330,60\nEQ-0068,Orion 15,Harbor Devices,2026-09-12T12:00:00Z,77,0.10,940,1320,370,150,130,60\nEQ-0069,Lumen 13,Harbor Devices,2026-09-13T12:00:00Z,10,0.57,650,1110,580,120,640,60\nEQ-0070,Atlas 14,Meridian Systems,2026-09-14T12:00:00Z,20,0.08,1040,1460,420,100,140,90\nEQ-0071,Orion 15,Meridian Systems,2026-09-15T12:00:00Z,56,0.33,960,1340,600,150,380,70\nEQ-0072,Lumen 13,Meridian Systems,2026-09-16T12:00:00Z,46,0.51,710,1190,210,120,390,30\nEQ-0073,Atlas 14,Northstar IT,2026-09-17T12:00:00Z,52,0.42,1020,1420,290,100,420,40\nEQ-0074,Orion 15,Northstar IT,2026-09-18T12:00:00Z,53,0.27,960,1470,300,150,440,90\nEQ-0075,Lumen 13,Northstar IT,2026-09-19T12:00:00Z,23,0.07,750,1060,580,120,530,70\nEQ-0076,Atlas 14,Harbor Devices,2026-09-20T12:00:00Z,47,0.44,1100,1490,580,100,330,50\nEQ-0077,Orion 15,Harbor Devices,2026-09-21T12:00:00Z,75,0.52,860,1430,680,150,600,30\nEQ-0078,Lumen 13,Harbor Devices,2026-09-22T12:00:00Z,78,0.04,700,1020,210,120,580,70\nEQ-0079,Atlas 14,Meridian Systems,2026-09-23T12:00:00Z,70,0.34,1050,1400,790,100,470,40\nEQ-0080,Orion 15,Meridian Systems,2026-09-24T12:00:00Z,74,0.16,1000,1310,730,150,540,80\nEQ-0081,Lumen 13,Meridian Systems,2026-09-25T12:00:00Z,81,0.19,730,1030,730,120,250,60\nEQ-0082,Atlas 14,Northstar IT,2026-09-26T12:00:00Z,28,0.42,1020,1380,190,100,470,60\nEQ-0083,Orion 15,Northstar IT,2026-09-27T12:00:00Z,58,0.20,950,1420,130,150,650,30\nEQ-0084,Lumen 13,Northstar IT,2026-09-28T12:00:00Z,73,0.24,610,1120,820,120,50,90\nEQ-0085,Atlas 14,Harbor Devices,2026-09-01T12:00:00Z,95,0.58,900,1310,210,100,200,70\nEQ-0086,Orion 15,Harbor Devices,2026-09-02T12:00:00Z,22,0.19,850,1300,470,150,490,60\nEQ-0087,Lumen 13,Harbor Devices,2026-09-03T12:00:00Z,73,0.35,740,1060,520,120,630,40\nEQ-0088,Atlas 14,Meridian Systems,2026-09-04T12:00:00Z,21,0.46,930,1360,770,100,610,60\nEQ-0089,Orion 15,Meridian Systems,2026-09-05T12:00:00Z,99,0.52,820,1380,470,150,190,30\nEQ-0090,Lumen 13,Meridian Systems,2026-09-06T12:00:00Z,71,0.58,760,1100,80,120,130,90\nEQ-0091,Atlas 14,Northstar IT,2026-09-07T12:00:00Z,34,0.03,1030,1360,210,100,590,70\nEQ-0092,Orion 15,Northstar IT,2026-09-08T12:00:00Z,69,0.39,850,1390,310,150,310,60\nEQ-0093,Lumen 13,Northstar IT,2026-09-09T12:00:00Z,62,0.58,740,1000,170,120,140,50\nEQ-0094,Atlas 14,Harbor Devices,2026-09-10T12:00:00Z,97,0.13,920,1430,380,100,160,60\nEQ-0095,Orion 15,Harbor Devices,2026-09-11T12:00:00Z,61,0.16,910,1450,600,150,250,50\nEQ-0096,Lumen 13,Harbor Devices,2026-09-12T12:00:00Z,10,0.18,710,1010,790,120,460,90\nEQ-0097,Atlas 14,Meridian Systems,2026-09-13T12:00:00Z,49,0.55,990,1500,350,100,550,90\nEQ-0098,Orion 15,Meridian Systems,2026-09-14T12:00:00Z,32,0.38,820,1470,800,150,180,30\nEQ-0099,Lumen 13,Meridian Systems,2026-09-15T12:00:00Z,59,0.03,730,1000,630,120,300,50\nEQ-0100,Atlas 14,Northstar IT,2026-09-16T12:00:00Z,57,0.51,1080,1480,770,100,230,80\nEQ-0101,Orion 15,Northstar IT,2026-09-17T12:00:00Z,74,0.59,810,1460,540,150,650,70\nEQ-0102,Lumen 13,Northstar IT,2026-09-18T12:00:00Z,26,0.58,630,1000,510,120,240,80\nEQ-0103,Atlas 14,Harbor Devices,2026-09-19T12:00:00Z,31,0.32,950,1360,510,100,170,40\nEQ-0104,Orion 15,Harbor Devices,2026-09-20T12:00:00Z,40,0.43,990,1410,840,150,130,30\nEQ-0105,Lumen 13,Harbor Devices,2026-09-21T12:00:00Z,39,0.29,690,1020,750,120,260,30\nEQ-0106,Atlas 14,Meridian Systems,2026-09-22T12:00:00Z,62,0.03,970,1320,570,100,380,80\nEQ-0107,Orion 15,Meridian Systems,2026-09-23T12:00:00Z,48,0.05,970,1300,530,150,490,50\nEQ-0108,Lumen 13,Meridian Systems,2026-09-24T12:00:00Z,25,0.09,740,1200,160,120,130,40\nEQ-0109,Atlas 14,Northstar IT,2026-09-25T12:00:00Z,94,0.16,920,1340,220,100,560,50\nEQ-0110,Orion 15,Northstar IT,2026-09-26T12:00:00Z,66,0.51,950,1500,750,150,320,90\nEQ-0111,Lumen 13,Northstar IT,2026-09-27T12:00:00Z,66,0.25,800,1040,830,120,280,50\nEQ-0112,Atlas 14,Harbor Devices,2026-09-28T12:00:00Z,85,0.09,910,1410,780,100,480,50\nEQ-0113,Orion 15,Harbor Devices,2026-09-01T12:00:00Z,54,0.21,910,1420,680,150,50,30\nEQ-0114,Lumen 13,Harbor Devices,2026-09-02T12:00:00Z,15,0.26,630,1160,830,120,260,60\nEQ-0115,Atlas 14,Meridian Systems,2026-09-03T12:00:00Z,79,0.60,1050,1460,320,100,370,50\nEQ-0116,Orion 15,Meridian Systems,2026-09-04T12:00:00Z,91,0.41,860,1360,130,150,510,30\nEQ-0117,Lumen 13,Meridian Systems,2026-09-05T12:00:00Z,41,0.56,790,1170,680,120,640,90\nEQ-0118,Atlas 14,Northstar IT,2026-09-06T12:00:00Z,87,0.52,1010,1440,710,100,330,50\nEQ-0119,Orion 15,Northstar IT,2026-09-07T12:00:00Z,60,0.20,840,1340,580,150,310,70\nEQ-0120,Lumen 13,Northstar IT,2026-09-08T12:00:00Z,70,0.39,800,1190,490,120,240,40\nEQ-0121,Atlas 14,Harbor Devices,2026-09-09T12:00:00Z,15,0.41,1100,1410,680,100,230,30\nEQ-0122,Orion 15,Harbor Devices,2026-09-10T12:00:00Z,68,0.23,970,1380,810,150,290,80\nEQ-0123,Lumen 13,Harbor Devices,2026-09-11T12:00:00Z,10,0.60,640,1100,160,120,450,80\nEQ-0124,Atlas 14,Meridian Systems,2026-09-12T12:00:00Z,91,0.31,930,1300,490,100,290,30\nEQ-0125,Orion 15,Meridian Systems,2026-09-13T12:00:00Z,11,0.13,870,1350,600,150,240,30\nEQ-0126,Lumen 13,Meridian Systems,2026-09-14T12:00:00Z,26,0.36,610,1110,340,120,500,30\nEQ-0127,Atlas 14,Northstar IT,2026-09-15T12:00:00Z,30,0.53,1040,1460,260,100,560,40\nEQ-0128,Orion 15,Northstar IT,2026-09-16T12:00:00Z,56,0.54,1000,1370,350,150,130,70\nEQ-0129,Lumen 13,Northstar IT,2026-09-17T12:00:00Z,67,0.57,690,1110,630,120,350,50\nEQ-0130,Atlas 14,Harbor Devices,2026-09-18T12:00:00Z,27,0.10,920,1500,280,100,320,50\nEQ-0131,Orion 15,Harbor Devices,2026-09-19T12:00:00Z,16,0.22,880,1450,480,150,410,90\nEQ-0132,Lumen 13,Harbor Devices,2026-09-20T12:00:00Z,79,0.10,640,1080,100,120,190,60\nEQ-0133,Atlas 14,Meridian Systems,2026-09-21T12:00:00Z,82,0.24,900,1430,420,100,350,30\nEQ-0134,Orion 15,Meridian Systems,2026-09-22T12:00:00Z,100,0.48,860,1330,590,150,230,70\nEQ-0135,Lumen 13,Meridian Systems,2026-09-23T12:00:00Z,68,0.17,680,1060,400,120,560,90\nEQ-0136,Atlas 14,Northstar IT,2026-09-24T12:00:00Z,85,0.24,900,1440,200,100,370,70\nEQ-0137,Orion 15,Northstar IT,2026-09-25T12:00:00Z,60,0.11,940,1500,190,150,240,90\nEQ-0138,Lumen 13,Northstar IT,2026-09-26T12:00:00Z,99,0.11,710,1170,370,120,420,60\nEQ-0139,Atlas 14,Harbor Devices,2026-09-27T12:00:00Z,37,0.58,930,1460,780,100,200,60\nEQ-0140,Orion 15,Harbor Devices,2026-09-28T12:00:00Z,51,0.34,880,1370,550,150,40,50\nEQ-0141,Lumen 13,Harbor Devices,2026-09-01T12:00:00Z,97,0.43,680,1100,750,120,220,90\nEQ-0142,Atlas 14,Meridian Systems,2026-09-02T12:00:00Z,43,0.12,940,1450,120,100,280,80\nEQ-0143,Orion 15,Meridian Systems,2026-09-03T12:00:00Z,53,0.35,920,1390,560,150,200,30\nEQ-0144,Lumen 13,Meridian Systems,2026-09-04T12:00:00Z,42,0.16,770,1020,620,120,550,70\nEQ-0145,Atlas 14,Northstar IT,2026-09-05T12:00:00Z,20,0.39,1010,1310,340,100,560,50\nEQ-0146,Orion 15,Northstar IT,2026-09-06T12:00:00Z,48,0.21,960,1310,800,150,240,90\nEQ-0147,Lumen 13,Northstar IT,2026-09-07T12:00:00Z,19,0.21,630,1050,430,120,320,70\nEQ-0148,Atlas 14,Harbor Devices,2026-09-08T12:00:00Z,17,0.24,970,1350,750,100,540,60\nEQ-0149,Orion 15,Harbor Devices,2026-09-09T12:00:00Z,86,0.48,830,1490,760,150,140,90\nEQ-0150,Lumen 13,Harbor Devices,2026-09-10T12:00:00Z,19,0.39,720,1110,800,120,80,50\nEQ-0151,Atlas 14,Meridian Systems,2026-09-11T12:00:00Z,79,0.05,1080,1360,100,100,310,50\nEQ-0152,Orion 15,Meridian Systems,2026-09-12T12:00:00Z,75,0.22,920,1410,280,150,370,50\nEQ-0153,Lumen 13,Meridian Systems,2026-09-13T12:00:00Z,18,0.28,800,1200,480,120,460,70\nEQ-0154,Atlas 14,Northstar IT,2026-09-14T12:00:00Z,13,0.48,1020,1440,620,100,160,80\nEQ-0155,Orion 15,Northstar IT,2026-09-15T12:00:00Z,39,0.28,970,1430,160,150,370,80\nEQ-0156,Lumen 13,Northstar IT,2026-09-16T12:00:00Z,48,0.33,760,1090,650,120,360,70\nEQ-0157,Atlas 14,Harbor Devices,2026-09-17T12:00:00Z,41,0.60,930,1460,190,100,260,40\nEQ-0158,Orion 15,Harbor Devices,2026-09-18T12:00:00Z,90,0.25,840,1480,180,150,340,80\nEQ-0159,Lumen 13,Harbor Devices,2026-09-19T12:00:00Z,26,0.41,750,1040,720,120,170,60\nEQ-0160,Atlas 14,Meridian Systems,2026-09-20T12:00:00Z,36,0.11,950,1320,670,100,570,80\nEQ-0161,Orion 15,Meridian Systems,2026-09-21T12:00:00Z,90,0.27,920,1320,170,150,410,30\nEQ-0162,Lumen 13,Meridian Systems,2026-09-22T12:00:00Z,17,0.39,770,1060,280,120,540,60\nEQ-0163,Atlas 14,Northstar IT,2026-09-23T12:00:00Z,34,0.55,920,1330,100,100,590,70\nEQ-0164,Orion 15,Northstar IT,2026-09-24T12:00:00Z,33,0.29,990,1420,250,150,70,90\nEQ-0165,Lumen 13,Northstar IT,2026-09-25T12:00:00Z,21,0.35,690,1040,560,120,120,90\nEQ-0166,Atlas 14,Harbor Devices,2026-09-26T12:00:00Z,23,0.13,950,1500,350,100,270,70\nEQ-0167,Orion 15,Harbor Devices,2026-09-27T12:00:00Z,44,0.31,850,1430,90,150,580,90\nEQ-0168,Lumen 13,Harbor Devices,2026-09-28T12:00:00Z,33,0.14,730,1050,610,120,290,50\nEQ-0169,Atlas 14,Meridian Systems,2026-09-01T12:00:00Z,65,0.05,940,1360,160,100,480,70\nEQ-0170,Orion 15,Meridian Systems,2026-09-02T12:00:00Z,10,0.60,1000,1460,830,150,420,90\nEQ-0171,Lumen 13,Meridian Systems,2026-09-03T12:00:00Z,36,0.55,710,1130,190,120,410,60\nEQ-0172,Atlas 14,Northstar IT,2026-09-04T12:00:00Z,37,0.05,1090,1490,340,100,160,50\nEQ-0173,Orion 15,Northstar IT,2026-09-05T12:00:00Z,98,0.04,810,1500,650,150,150,90\nEQ-0174,Lumen 13,Northstar IT,2026-09-06T12:00:00Z,20,0.12,760,1040,390,120,500,40\nEQ-0175,Atlas 14,Harbor Devices,2026-09-07T12:00:00Z,97,0.17,1000,1470,540,100,140,90\nEQ-0176,Orion 15,Harbor Devices,2026-09-08T12:00:00Z,89,0.55,930,1340,430,150,290,70\nEQ-0177,Lumen 13,Harbor Devices,2026-09-09T12:00:00Z,31,0.48,800,1070,490,120,120,90\nEQ-0178,Atlas 14,Meridian Systems,2026-09-10T12:00:00Z,32,0.07,900,1320,750,100,610,60\nEQ-0179,Orion 15,Meridian Systems,2026-09-11T12:00:00Z,48,0.02,990,1320,830,150,70,70\nEQ-0180,Lumen 13,Meridian Systems,2026-09-12T12:00:00Z,73,0.51,750,1180,830,120,40,80\nEQ-0181,Atlas 14,Northstar IT,2026-09-13T12:00:00Z,23,0.12,1020,1370,330,100,50,80\nEQ-0182,Orion 15,Northstar IT,2026-09-14T12:00:00Z,33,0.38,910,1480,500,150,340,80\nEQ-0183,Lumen 13,Northstar IT,2026-09-15T12:00:00Z,95,0.12,630,1120,450,120,450,70\nEQ-0184,Atlas 14,Harbor Devices,2026-09-16T12:00:00Z,50,0.21,1070,1480,470,100,90,50\nEQ-0185,Orion 15,Harbor Devices,2026-09-17T12:00:00Z,84,0.49,930,1450,680,150,130,70\nEQ-0186,Lumen 13,Harbor Devices,2026-09-18T12:00:00Z,60,0.59,740,1140,660,120,370,30\nEQ-0187,Atlas 14,Meridian Systems,2026-09-19T12:00:00Z,75,0.59,990,1330,500,100,640,50\nEQ-0188,Orion 15,Meridian Systems,2026-09-20T12:00:00Z,32,0.02,1000,1370,670,150,130,70\nEQ-0189,Lumen 13,Meridian Systems,2026-09-21T12:00:00Z,74,0.05,600,1030,620,120,470,60\nEQ-0190,Atlas 14,Northstar IT,2026-09-22T12:00:00Z,93,0.14,1030,1390,490,100,130,60\nEQ-0191,Orion 15,Northstar IT,2026-09-23T12:00:00Z,63,0.55,980,1460,220,150,510,90\nEQ-0192,Lumen 13,Northstar IT,2026-09-24T12:00:00Z,40,0.44,790,1060,550,120,150,50\nEQ-0193,Atlas 14,Harbor Devices,2026-09-25T12:00:00Z,59,0.54,950,1320,580,100,450,30\nEQ-0194,Orion 15,Harbor Devices,2026-09-26T12:00:00Z,27,0.52,990,1500,130,150,550,70\nEQ-0195,Lumen 13,Harbor Devices,2026-09-27T12:00:00Z,67,0.11,800,1190,300,120,560,80\nEQ-0196,Atlas 14,Meridian Systems,2026-09-28T12:00:00Z,80,0.59,1020,1350,490,100,140,90\nEQ-0197,Orion 15,Meridian Systems,2026-09-01T12:00:00Z,46,0.39,890,1410,730,150,200,50\nEQ-0198,Lumen 13,Meridian Systems,2026-09-02T12:00:00Z,11,0.20,680,1120,810,120,640,90\nEQ-0199,Atlas 14,Northstar IT,2026-09-03T12:00:00Z,33,0.31,1030,1460,840,100,530,40\nEQ-0200,Orion 15,Northstar IT,2026-09-04T12:00:00Z,16,0.08,980,1340,160,150,350,60\nEQ-0201,Lumen 13,Northstar IT,2026-09-05T12:00:00Z,72,0.06,670,1120,230,120,40,90\nEQ-0202,Atlas 14,Harbor Devices,2026-09-06T12:00:00Z,47,0.39,990,1450,800,100,590,60\nEQ-0203,Orion 15,Harbor Devices,2026-09-07T12:00:00Z,85,0.09,890,1380,520,150,320,40\nEQ-0204,Lumen 13,Harbor Devices,2026-09-08T12:00:00Z,62,0.32,800,1020,90,120,180,70\nEQ-0205,Atlas 14,Meridian Systems,2026-09-09T12:00:00Z,31,0.27,1050,1410,600,100,490,70\nEQ-0206,Orion 15,Meridian Systems,2026-09-10T12:00:00Z,81,0.59,820,1460,140,150,380,90\nEQ-0207,Lumen 13,Meridian Systems,2026-09-11T12:00:00Z,13,0.11,790,1120,830,120,330,70\nEQ-0208,Atlas 14,Northstar IT,2026-09-12T12:00:00Z,52,0.36,930,1340,610,100,380,70\nEQ-0209,Orion 15,Northstar IT,2026-09-13T12:00:00Z,74,0.11,800,1330,740,150,490,60\nEQ-0210,Lumen 13,Northstar IT,2026-09-14T12:00:00Z,97,0.22,770,1200,810,120,160,60\nEQ-0211,Atlas 14,Harbor Devices,2026-09-15T12:00:00Z,85,0.45,970,1470,130,100,190,50\nEQ-0212,Orion 15,Harbor Devices,2026-09-16T12:00:00Z,57,0.46,910,1470,190,150,630,90\nEQ-0213,Lumen 13,Harbor Devices,2026-09-17T12:00:00Z,73,0.31,700,1050,310,120,400,70\nEQ-0214,Atlas 14,Meridian Systems,2026-09-18T12:00:00Z,67,0.03,1090,1440,780,100,170,70\nEQ-0215,Orion 15,Meridian Systems,2026-09-19T12:00:00Z,88,0.30,800,1380,670,150,260,70\nEQ-0216,Lumen 13,Meridian Systems,2026-09-20T12:00:00Z,28,0.31,770,1160,170,120,390,70\nEQ-0217,Atlas 14,Northstar IT,2026-09-21T12:00:00Z,74,0.23,1040,1460,430,100,90,50\nEQ-0218,Orion 15,Northstar IT,2026-09-22T12:00:00Z,15,0.29,850,1340,540,150,190,60\nEQ-0219,Lumen 13,Northstar IT,2026-09-23T12:00:00Z,73,0.39,630,1000,120,120,240,90\nEQ-0220,Atlas 14,Harbor Devices,2026-09-24T12:00:00Z,96,0.50,1090,1330,660,100,450,70\nEQ-0221,Orion 15,Harbor Devices,2026-09-25T12:00:00Z,89,0.47,830,1410,130,150,440,70\nEQ-0222,Lumen 13,Harbor Devices,2026-09-26T12:00:00Z,86,0.60,620,1180,770,120,420,90\nEQ-0223,Atlas 14,Meridian Systems,2026-09-27T12:00:00Z,45,0.47,1040,1310,590,100,190,40\nEQ-0224,Orion 15,Meridian Systems,2026-09-28T12:00:00Z,80,0.44,800,1380,790,150,350,60\nEQ-0225,Lumen 13,Meridian Systems,2026-09-01T12:00:00Z,77,0.37,790,1040,450,120,360,80\nEQ-0226,Atlas 14,Northstar IT,2026-09-02T12:00:00Z,34,0.52,1050,1480,400,100,80,50\nEQ-0227,Orion 15,Northstar IT,2026-09-03T12:00:00Z,18,0.19,930,1300,480,150,80,50\nEQ-0228,Lumen 13,Northstar IT,2026-09-04T12:00:00Z,65,0.42,680,1080,360,120,70,40\nEQ-0229,Atlas 14,Harbor Devices,2026-09-05T12:00:00Z,84,0.21,980,1380,280,100,360,50\nEQ-0230,Orion 15,Harbor Devices,2026-09-06T12:00:00Z,54,0.28,870,1300,410,150,70,90\nEQ-0231,Lumen 13,Harbor Devices,2026-09-07T12:00:00Z,98,0.07,660,1200,640,120,150,30\nEQ-0232,Atlas 14,Meridian Systems,2026-09-08T12:00:00Z,61,0.52,970,1440,460,100,320,90\nEQ-0233,Orion 15,Meridian Systems,2026-09-09T12:00:00Z,71,0.42,880,1410,740,150,360,30\nEQ-0234,Lumen 13,Meridian Systems,2026-09-10T12:00:00Z,60,0.02,750,1170,600,120,490,90\nEQ-0235,Atlas 14,Northstar IT,2026-09-11T12:00:00Z,84,0.60,960,1380,660,100,190,50\nEQ-0236,Orion 15,Northstar IT,2026-09-12T12:00:00Z,77,0.13,890,1470,230,150,390,70\nEQ-0237,Lumen 13,Northstar IT,2026-09-13T12:00:00Z,53,0.06,700,1030,400,120,340,40\nEQ-0238,Atlas 14,Harbor Devices,2026-09-14T12:00:00Z,83,0.48,990,1350,750,100,260,60\nEQ-0239,Orion 15,Harbor Devices,2026-09-15T12:00:00Z,27,0.18,960,1440,390,150,570,40\nEQ-0240,Lumen 13,Harbor Devices,2026-09-16T12:00:00Z,87,0.44,780,1160,700,120,270,30\nDEMO-001,Atlas 14,Northstar IT,2026-09-30T12:00:00Z,85,0.1,1000,1400,200,100,400,50\nDEMO-002,Orion 15,Harbor Devices,2026-09-30T12:00:00Z,65,0.55,900,1400,850,150,600,70\nDEMO-003,Lumen 13,Northstar IT,2026-09-30T12:00:00Z,20,0.3,700,1100,250,120,40,30\nEQ-0001,Atlas 14,Northstar IT,2026-09-01T12:00:00Z,45,0.38,930,1460,690,100,150,60\nEQ-0002,Orion 15,Northstar IT,2026-09-02T12:00:00Z,48,0.27,980,1320,700,150,370,70\nEQ-0003,Lumen 13,Northstar IT,2026-09-03T12:00:00Z,35,0.18,740,1030,100,120,300,90\nEQ-0004,Atlas 14,Harbor Devices,2026-09-04T12:00:00Z,13,0.29,1030,1410,790,100,350,70\nEQ-0005,Orion 15,Harbor Devices,2026-09-05T12:00:00Z,65,0.38,850,1310,410,150,420,40\nEQ-0006,Lumen 13,Harbor Devices,2026-09-06T12:00:00Z,21,0.10,740,1110,450,120,180,30\nEQ-0007,Atlas 14,Meridian Systems,2026-09-07T12:00:00Z,37,0.50,980,1380,140,100,580,40\nEQ-0008,Orion 15,Meridian Systems,2026-09-08T12:00:00Z,74,0.07,850,1490,630,150,90,90\nEQ-0001,Atlas 14,Northstar IT,2026-09-30T12:00:00Z,45,0.38,930,1460,800,100,150,60\nEQ-0001,Atlas 14,Northstar IT,2026-08-01T12:00:00Z,45,0.38,930,1460,50,100,150,60\nBAD-001,Atlas 14,Northstar IT,2026-09-01T12:00:00Z,45,1.2,930,1460,690,100,150,60\nBAD-002,Atlas 14,Northstar IT,2026-09-01T12:00:00Z,45,0.38,930,1460,-10,100,150,60\nBAD-003,Atlas 14,Northstar IT,2026-09-01T12:00:00Z,101,0.38,930,1460,690,100,150,60\nBAD-004,,Northstar IT,2026-09-01T12:00:00Z,45,0.38,930,1460,690,100,150,60\nBAD-005,Atlas 14,Northstar IT,not-a-date,45,0.38,930,1460,690,100,150,60\nBAD-006,Atlas 14,Northstar IT,2026-09-01T12:00:00Z,45,0.38,unknown,1460,690,100,150,60\nBAD-007,Atlas 14,Northstar IT,2026-09-01T12:00:00Z,69.5,0.38,930,1460,690,100,150,60\n\n'
default_input_path = volume_root + '/equipment_events.csv'
dbutils.fs.put(default_input_path, fixture_csv, overwrite=True)
print("Synthetic fixture ready:", default_input_path)


# Circular Intelligence · a risk-aware recovery lakehouse
All values are synthetic BRL assumptions. No actual environmental impact is claimed.
Upload data/equipment_events.csv to a Unity Catalog volume. Supply its full /Volumes/... path.
The execution principal needs USE CATALOG, USE/CREATE SCHEMA, CREATE TABLE,
SELECT/MODIFY on output tables, and READ VOLUME on the source.

In [ ]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType
from delta.tables import DeltaTable
import re

dbutils.widgets.text("catalog", default_catalog)
dbutils.widgets.text("schema", "circular_portfolio")
dbutils.widgets.text("input_path", default_input_path)
catalog = dbutils.widgets.get("catalog")
schema_name = dbutils.widgets.get("schema")
input_path = dbutils.widgets.get("input_path")
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog), "Use a simple catalog identifier"
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", schema_name), "Use a simple schema identifier"
assert input_path.startswith("/Volumes/") and input_path.endswith(".csv"), "Expected a UC volume CSV path"
spark.conf.set("spark.sql.session.timeZone", "UTC")
prefix = f"{catalog}.{schema_name}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {prefix}")
columns = ["equipment_id","model","supplier","updated_at","condition_score","failure_probability",
           "resale_price","refurbished_price","repair_cost","salvage_value","acquisition_credit","logistics_cost"]
money_columns = columns[6:]

def upsert_events(frame, table):
    if not spark.catalog.tableExists(table):
        frame.write.format("delta").mode("append").saveAsTable(table)
    else:
        (DeltaTable.forName(spark, table).alias("t").merge(frame.alias("s"), "t.event_hash = s.event_hash")
         .whenNotMatchedInsertAll().execute())


In [ ]:
# Bronze preserves raw values. A hash of the ordered source fields makes replays idempotent.
raw_schema = StructType([StructField(c, StringType(), True) for c in columns])
source = (spark.read.option("header", True).option("mode", "FAILFAST").schema(raw_schema).csv(input_path))
bronze_input = (source.withColumn("event_hash", F.sha2(F.to_json(F.struct(*[F.col(c) for c in columns])),256))
               .withColumn("source_file", F.lit(input_path)).withColumn("ingested_at", F.current_timestamp())
               .dropDuplicates(["event_hash"]))
upsert_events(bronze_input, f"{prefix}.bronze_equipment_events")


In [ ]:
# Silver normalization scans the demo Bronze table on each run.
# Ingestion is incremental; this is deliberately not advertised as incremental Silver processing.
bronze = spark.table(f"{prefix}.bronze_equipment_events")
typed = bronze
for c in money_columns:
    typed = typed.withColumn(c + "_value", F.expr(f"try_cast({c} AS DECIMAL(20,6))"))
typed = (typed.withColumn("condition_value", F.expr("try_cast(condition_score AS DECIMAL(10,4))"))
         .withColumn("probability_value", F.expr("try_cast(failure_probability AS DECIMAL(10,6))"))
         .withColumn("event_time", F.expr("try_cast(updated_at AS TIMESTAMP)")))
missing = F.lit(False)
for c in columns:
    missing = missing | F.col(c).isNull() | (F.trim(F.col(c)) == "")
invalid_money = F.lit(False)
for c in money_columns:
    v = F.col(c + "_value")
    invalid_money = invalid_money | v.isNull() | (v < 0) | (v != F.round(v,2)) | ~F.trim(F.col(c)).rlike(r"^\d{1,14}(\.\d{1,2})?$")
condition = F.col("condition_value")
probability = F.col("probability_value")
invalid_condition = condition.isNull() | (condition < 0) | (condition > 100) | (condition != F.floor(condition)) | ~F.trim(F.col("condition_score")).rlike(r"^\d{1,3}$")
invalid_probability = probability.isNull() | (probability < 0) | (probability > 1) | (probability != F.round(probability,4)) | ~F.trim(F.col("failure_probability")).rlike(r"^\d(\.\d{1,4})?$")
invalid_timestamp = F.col("event_time").isNull() | ~F.col("updated_at").rlike(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(Z|\+00:00)$")
typed = typed.withColumn("quality_reason", F.when(missing,"missing_fields")
    .when(invalid_money,"invalid_money").when(invalid_condition,"invalid_condition")
    .when(invalid_probability,"invalid_probability").when(invalid_timestamp,"invalid_timestamp"))
quarantine = typed.filter(F.col("quality_reason").isNotNull())
upsert_events(quarantine, f"{prefix}.quarantine_equipment_events")
valid = typed.filter(F.col("quality_reason").isNull())
for c in ("equipment_id","model","supplier"):
    valid = valid.withColumn(c,F.trim(F.col(c)))
window = Window.partitionBy("equipment_id").orderBy(F.col("event_time").desc(),F.col("event_hash").desc())
latest = valid.withColumn("_rank",F.row_number().over(window)).filter("_rank = 1")
silver = latest.select("equipment_id","model","supplier","updated_at","event_time","event_hash",
    F.col("condition_value").cast("int").alias("condition_score"),
    F.col("probability_value").cast("decimal(5,4)").alias("failure_probability"),
    *[F.col(c+"_value").cast("decimal(16,2)").alias(c) for c in money_columns])
if not spark.catalog.tableExists(f"{prefix}.silver_equipment"):
    silver.write.format("delta").mode("append").saveAsTable(f"{prefix}.silver_equipment")
else:
    (DeltaTable.forName(spark,f"{prefix}.silver_equipment").alias("t")
     .merge(silver.alias("s"),"t.equipment_id = s.equipment_id")
     .whenMatchedUpdateAll(condition="s.event_time > t.event_time OR (s.event_time = t.event_time AND s.event_hash > t.event_hash)")
     .whenNotMatchedInsertAll().execute())


In [ ]:
# Compare unrounded candidate values before rounding currency for presentation.
s = spark.table(f"{prefix}.silver_equipment")
base = F.col("acquisition_credit") + F.col("logistics_cost")
scored = (s.withColumn("resale_raw",F.col("resale_price")-base)
    .withColumn("repair_raw",(1-F.col("failure_probability"))*F.col("refurbished_price") +
        F.col("failure_probability")*F.col("salvage_value")-F.col("repair_cost")-base)
    .withColumn("recycle_raw",F.col("salvage_value")-base)
    .withColumn("naive_repair_raw",F.col("refurbished_price")-F.col("repair_cost")-base))

def choose_route(repair_column):
    resale = F.col("resale_raw")
    repair = F.col(repair_column)
    recycle = F.col("recycle_raw")
    condition = F.col("condition_score")
    return (F.when((condition>=70)&(resale>=repair)&(resale>=recycle),"Resale")
        .when((condition>=30)&(repair>=recycle)&((condition<70)|(repair>resale)),"Repair")
        .when((condition>=70)&(resale>=recycle),"Resale").otherwise("Recycle"))

def selected_value(route_column):
    return (F.when(F.col(route_column)=="Resale",F.col("resale_raw"))
        .when(F.col(route_column)=="Repair",F.col("repair_raw")).otherwise(F.col("recycle_raw")))

decisions = scored.withColumn("route",choose_route("repair_raw")).withColumn("naive_route",choose_route("naive_repair_raw"))
decisions = (decisions.withColumn("expected_net",F.round(selected_value("route"),2))
    .withColumn("baseline_expected_net",F.round(selected_value("naive_route"),2))
    .withColumn("decision_gain",F.round(selected_value("route")-selected_value("naive_route"),2))
    .withColumn("needs_review",selected_value("route")<0)
    .withColumn("policy_version",F.lit("circular-v1")))
gold = decisions.select("equipment_id","model","supplier","condition_score","route","naive_route",
    "expected_net","baseline_expected_net","decision_gain","needs_review","policy_version","event_time",
    F.round("resale_raw",2).alias("resale_net"),F.round("repair_raw",2).alias("repair_net"),
    F.round("recycle_raw",2).alias("recycle_net"))
gold.write.format("delta").mode("overwrite").option("overwriteSchema",True).saveAsTable(f"{prefix}.gold_asset_decisions")
model_summary = gold.groupBy("model").agg(F.count("*").alias("assets"),F.sum("expected_net").alias("expected_net"),
    F.sum(F.col("needs_review").cast("int")).alias("review_assets"),F.sum("decision_gain").alias("decision_gain"))
model_summary.write.format("delta").mode("overwrite").option("overwriteSchema",True).saveAsTable(f"{prefix}.gold_model_summary")


In [ ]:
# Fail the run if key integrity or policy invariants break. Replays should keep these stable.
assert silver.count() == silver.select("equipment_id").distinct().count(), "Duplicate Silver equipment IDs"
assert gold.filter("decision_gain < 0").count() == 0, "Policy underperforms its own eligible baseline"
assert gold.filter("route = 'Resale' AND condition_score < 70").count() == 0, "Ineligible resale"
assert gold.filter("route = 'Repair' AND condition_score < 30").count() == 0, "Ineligible repair"
display(model_summary.orderBy(F.col("expected_net").desc()))
display(gold.filter("needs_review").orderBy("expected_net"))
display(quarantine.select("equipment_id","quality_reason"))


In [ ]:
from decimal import Decimal
assert spark.table(f"{prefix}.bronze_equipment_events").count() == 252
assert spark.table(f"{prefix}.quarantine_equipment_events").count() == 7
assert spark.table(f"{prefix}.silver_equipment").count() == 243
totals = spark.table(f"{prefix}.gold_asset_decisions").agg(F.sum("expected_net").alias("expected"), F.sum("decision_gain").alias("gain")).first()
assert totals.expected == Decimal("33861.40"), totals
assert totals.gain == Decimal("9066.40"), totals
print("CIRCULAR FIXTURE VALIDATED IN DATABRICKS", totals.asDict())
